# <font color="#418FDE" size="6.5" uppercase>**Responsible Deployment**</font>

>Last update: 20260709.
    
By the end of this Lecture, you will be able to:
- Evaluate competing models using metrics, plots, interpretability, and engineering consequences. 
- Identify risks involving bias, imbalance, explainability, data drift, and unsafe automation. 
- Design a responsible final machine learning workflow for a civil engineering use case. 


## **1. Responsible Model Selection**

### **1.1. Model Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_08/Lecture_B/image_01_01.jpg?v=1783626971" width="250">



>* Metrics reveal some errors while hiding others
>* Choose metrics based on engineering consequences

>* Choose metrics for the model task
>* Check critical errors, not just averages

>* Compare metrics across plots, thresholds, and subgroups
>* Balance model evidence with engineering consequences



In [ ]:
#@title Python Code - Model Metrics

# Compare metrics for responsible model selection.
# Civil engineering errors have different consequences.
# Threshold choices change inspection workload and risk.

import numpy as np
import matplotlib.pyplot as plt

# Store true bridge deck defect labels.
y_true = np.array([
    0, 0, 0, 0, 0, 0,
    0, 0, 0, 0, 1, 1,
    1, 1, 1, 1, 1, 1
])

# Store two competing model risk scores.
model_a = np.array([
    .05, .10, .12, .18, .20, .25,
    .30, .35, .40, .45, .42, .48,
    .55, .60, .65, .70, .75, .80
])

model_b = np.array([
    .02, .04, .06, .08, .10, .12,
    .14, .16, .18, .20, .30, .35,
    .40, .45, .50, .55, .60, .65
])

# Confirm both score arrays match labels.
assert y_true.shape == model_a.shape == model_b.shape

# Define a compact metric calculator.
def metrics_at_threshold(scores, threshold):
    preds = (scores >= threshold).astype(int)
    tp = int(((preds == 1) & (y_true == 1)).sum())
    tn = int(((preds == 0) & (y_true == 0)).sum())

    fp = int(((preds == 1) & (y_true == 0)).sum())
    fn = int(((preds == 0) & (y_true == 1)).sum())
    accuracy = (tp + tn) / len(y_true)
    recall = tp / max(tp + fn, 1)

    precision = tp / max(tp + fp, 1)
    return accuracy, recall, precision, fp, fn

# Evaluate both models at one alert threshold.
threshold = 0.50
results = {
    "Model A": metrics_at_threshold(model_a, threshold),
    "Model B": metrics_at_threshold(model_b, threshold),
}

# Print concise metric evidence for comparison.
print("Responsible model metrics at threshold 0.50")
for name, values in results.items():
    acc, rec, prec, fp, fn = values
    print(f"{name}: accuracy={acc:.2f}, recall={rec:.2f}, precision={prec:.2f}")

# Translate missed defects into engineering consequence.
for name, values in results.items():
    missed = values[4]
    print(f"{name}: missed serious defects = {missed}")

# Plot threshold tradeoffs for missed defects.
thresholds = np.linspace(0.20, 0.70, 11)
miss_a = [metrics_at_threshold(model_a, t)[4] for t in thresholds]
miss_b = [metrics_at_threshold(model_b, t)[4] for t in thresholds]

# Create one plot showing risk tradeoffs.
plt.figure(figsize=(7, 4))
plt.plot(thresholds, miss_a, marker="o", label="Model A")
plt.plot(thresholds, miss_b, marker="s", label="Model B")
plt.xlabel("Alert threshold for serious cracking")

plt.ylabel("Missed serious defects")
plt.title("Metric choice reveals engineering consequences")
plt.grid(True, alpha=0.3)
plt.legend()

# Display the single teaching plot.
plt.show()



### **1.2. Interpretability Insights**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_08/Lecture_B/image_01_02.jpg?v=1783626966" width="250">



>* Check models use meaningful engineering evidence
>* Prefer explainable reasoning when errors matter

>* Use multiple explanation methods to inspect behavior
>* Treat explanations as diagnostic evidence, not proof

>* Explain predictions for accountable stakeholder decisions
>* Balance model complexity with human oversight



In [ ]:
#@title Python Code - Interpretability Insights

# Interpretability helps engineers compare responsible model choices.
# This example uses transparent hand built scores.
# Feature contributions reveal possible shortcut reasoning.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create tiny bridge inspection cases.
data = pd.DataFrame({
    "crack_mm": [1, 4, 8, 12, 3, 15],
    "spall_cm2": [0, 20, 55, 80, 10, 95],
    "shadow_score": [8, 7, 2, 1, 9, 2],
    "severe": [0, 0, 1, 1, 0, 1],
})

# Define two competing scoring models.
weights_engineering = {"crack_mm": 0.55, "spall_cm2": 0.40, "shadow_score": 0.05}
weights_shortcut = {"crack_mm": 0.20, "spall_cm2": 0.15, "shadow_score": 0.65}

# Normalize features for fair contribution comparison.
features = data[["crack_mm", "spall_cm2", "shadow_score"]]
scaled = (features - features.min()) / (features.max() - features.min())

# Compute model scores from weighted feature evidence.
def score_cases(weight_dict):
    weight_series = pd.Series(weight_dict)
    return scaled.mul(weight_series).sum(axis=1)

# Convert scores into simple severe predictions.
engineering_scores = score_cases(weights_engineering)
shortcut_scores = score_cases(weights_shortcut)
engineering_pred = (engineering_scores >= 0.45).astype(int)
shortcut_pred = (shortcut_scores >= 0.45).astype(int)

# Calculate compact accuracy values.
engineering_acc = (engineering_pred == data["severe"]).mean()
shortcut_acc = (shortcut_pred == data["severe"]).mean()
print(f"Engineering model accuracy: {engineering_acc:.2f}")
print(f"Shortcut model accuracy: {shortcut_acc:.2f}")

# Explain one high risk case using contributions.
case_index = 3
eng_contrib = scaled.iloc[case_index] * pd.Series(weights_engineering)
short_contrib = scaled.iloc[case_index] * pd.Series(weights_shortcut)
print("Case 4 true label: severe deterioration")
print(f"Engineering top reason: {eng_contrib.idxmax()}")
print(f"Shortcut top reason: {short_contrib.idxmax()}")

# Plot contribution explanations for the selected case.
plot_data = pd.DataFrame({
    "Engineering model": eng_contrib,
    "Shortcut model": short_contrib,
})

# Compare whether explanations match engineering judgment.
ax = plot_data.plot(kind="bar", figsize=(7, 4))
ax.set_title("Interpretability check for one bridge deck case")
ax.set_ylabel("Contribution to severe score")
ax.set_xlabel("Input feature")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()



### **1.3. Operational Consequences**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_08/Lecture_B/image_01_03.jpg?v=1783626969" width="250">



>* Consider real-world impacts of model errors
>* Choose models aligned with operational priorities

>* Consider people, tools, and workflow demands
>* Balance accuracy with maintainability and integration

>* Plan for changing conditions and model drift
>* Use oversight to ensure responsible decisions



In [ ]:
#@title Python Code - Operational Consequences

# Compare models through operational consequences.
# Civil engineering errors have unequal costs.
# Simple metrics can hide field risk.

# Import compact numerical and plotting tools.
import numpy as np
import matplotlib.pyplot as plt

# Store confusion counts for two candidate models.
models = {
    "Model A": {"tp": 42, "fp": 18, "fn": 8, "tn": 132},
    "Model B": {"tp": 36, "fp": 6, "fn": 14, "tn": 144},
}

# Define operational costs in agency dollars.
cost_false_alarm = 1200
cost_missed_failure = 25000

# Prepare containers for summary values.
names = []
accuracies = []
operational_costs = []

# Compute accuracy and consequence cost.
for name, counts in models.items():
    total = sum(counts.values())
    correct = counts["tp"] + counts["tn"]
    accuracy = correct / total

    cost = counts["fp"] * cost_false_alarm
    cost += counts["fn"] * cost_missed_failure
    names.append(name)
    accuracies.append(accuracy)
    operational_costs.append(cost)

# Print a compact comparison table.
print("Responsible model selection example")
print("Model | Accuracy | Estimated consequence cost")
for name, acc, cost in zip(names, accuracies, operational_costs):
    print(f"{name} | {acc:.1%} | ${cost:,.0f}")

# Identify the lower consequence option.
best_index = int(np.argmin(operational_costs))
print(f"Lower operational risk: {names[best_index]}")

# Plot accuracy beside consequence cost.
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.bar(names, accuracies, color="steelblue", label="Accuracy")
ax1.set_ylim(0, 1)

# Add a second axis for dollars.
ax2 = ax1.twinx()
ax2.plot(names, operational_costs, color="darkred", marker="o")
ax2.set_ylabel("Estimated consequence cost, dollars")

# Label the teaching plot clearly.
ax1.set_ylabel("Accuracy")
ax1.set_title("Headline accuracy versus operational consequences")
fig.tight_layout()
plt.show()



## **2. Risk and Ethics**

### **2.1. Bias and Imbalance**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_08/Lecture_B/image_02_01.jpg?v=1783626973" width="250">



>* High accuracy can hide uneven model performance
>* Biased data can worsen infrastructure inequalities

>* Imbalance hides rare, local, or changing risks
>* Data reflects human choices and resource gaps

>* Check representation and subgroup performance
>* Mitigate risks before real-world deployment



In [ ]:
#@title Python Code - Bias and Imbalance

# Bias can hide behind average accuracy.
# Imbalanced inspection data can mislead engineers.
# Subgroup metrics reveal uneven model risk.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a small deterministic inspection dataset.
rng = np.random.default_rng(seed=42)
regions = ["Urban", "Rural", "Coastal"]
counts = [70, 20, 10]

# Store rows for simulated pavement inspections.
rows = []
for region, count in zip(regions, counts):
    for index in range(count):
        rows.append({"region": region})

# Convert rows into a compact dataframe.
data = pd.DataFrame(rows)
assert len(data) == sum(counts)

# Assign true severe crack rates by region.
rates = {"Urban": 0.10, "Rural": 0.25, "Coastal": 0.40}
data["severe"] = data["region"].map(rates)

# Sample true labels from regional risks.
random_values = rng.random(len(data))
data["true_defect"] = random_values < data["severe"]

# Simulate a biased detector with uneven sensitivity.
miss_rates = {"Urban": 0.15, "Rural": 0.45, "Coastal": 0.65}
false_alarm_rate = 0.05

# Generate predictions with more misses in underrepresented regions.
predictions = []
for _, row in data.iterrows():
    if row["true_defect"]:
        predictions.append(rng.random() > miss_rates[row["region"]])
    else:
        predictions.append(rng.random() < false_alarm_rate)

# Add predictions and compute overall accuracy.
data["predicted_defect"] = predictions
overall_accuracy = (data["true_defect"] == data["predicted_defect"]).mean()

# Compute subgroup recall for severe defects.
def recall_for_group(group):
    positives = group[group["true_defect"]]
    if len(positives) == 0:
        return np.nan
    return positives["predicted_defect"].mean()

# Summarize representation and safety performance.
summary = data.groupby("region").apply(recall_for_group)
summary = summary.rename("defect_recall").reset_index()
summary["sample_count"] = data.groupby("region").size().values

# Print concise teaching results.
print(f"Overall accuracy: {overall_accuracy:.2f}")
print("Subgroup recall shows hidden risk:")
for _, row in summary.iterrows():
    print(f"{row['region']}: n={int(row['sample_count'])}, recall={row['defect_recall']:.2f}")

# Plot subgroup recall against representation.
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.bar(summary["region"], summary["sample_count"], color="lightgray")
ax1.set_ylabel("Inspection images")
ax1.set_xlabel("Region")

# Add recall line on a second axis.
ax2 = ax1.twinx()
ax2.plot(summary["region"], summary["defect_recall"], marker="o", color="crimson")
ax2.set_ylabel("Recall for severe defects")
ax2.set_ylim(0, 1)

# Label the responsible deployment lesson.
plt.title("Imbalance can hide unsafe subgroup performance")
fig.tight_layout()
plt.show()



### **2.2. Bias and Imbalance**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_08/Lecture_B/image_02_02.jpg?v=1783626975" width="250">



>* Overall accuracy can hide uneven model performance
>* Biased data can create unfair infrastructure decisions

>* Rare safety cases can be underrepresented
>* Check performance across affected subgroups

>* Check data and subgroup performance carefully
>* Monitor ethical impacts throughout deployment



In [ ]:
#@title Python Code - Bias and Imbalance

# This example shows hidden subgroup risk.
# Overall accuracy can hide unsafe imbalance.
# Civil engineers must inspect subgroup performance.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a tiny inspection dataset.
np.random.seed(7)
urban_total, rural_total = 80, 20
urban_defects, rural_defects = 16, 10

# Store subgroup counts clearly.
data = pd.DataFrame({
    "road_group": ["Urban highways", "Rural roads"],
    "total_images": [urban_total, rural_total],
    "true_defects": [urban_defects, rural_defects],
    "missed_defects": [2, 7],
})

# Validate the small teaching dataset.
assert data["total_images"].min() > 0
assert (data["missed_defects"] <= data["true_defects"]).all()
data["detected_defects"] = data["true_defects"] - data["missed_defects"]

# Compute subgroup recall values.
data["recall"] = data["detected_defects"] / data["true_defects"]
overall_recall = data["detected_defects"].sum() / data["true_defects"].sum()
imbalance_ratio = data["total_images"].max() / data["total_images"].min()

# Print only concise teaching results.
print("Bias and imbalance check for pavement distress detection")
print(f"Overall defect recall: {overall_recall:.2f}")
print(f"Urban highway recall: {data.loc[0, 'recall']:.2f}")
print(f"Rural road recall: {data.loc[1, 'recall']:.2f}")
print(f"Image count imbalance ratio: {imbalance_ratio:.1f} to 1")
print("Ethics note: average performance hides rural safety risk.")

# Plot subgroup recall for comparison.
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(data["road_group"], data["recall"], color=["steelblue", "tomato"])
ax.axhline(overall_recall, color="black", linestyle="--", label="Overall recall")

# Label the engineering interpretation.
ax.set_ylim(0, 1.05)
ax.set_ylabel("Defect recall")
ax.set_title("Subgroup performance reveals imbalance risk")
ax.legend()

# Show the single required plot.
plt.tight_layout()
plt.show()



### **2.3. Unsafe Automation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_08/Lecture_B/image_02_03.jpg?v=1783626978" width="250">



>* Automation needs oversight and safeguards
>* Predictions should inform, not replace, engineers

>* Over-trusting models can reinforce infrastructure inequities
>* Using models beyond scope creates serious risks

>* Keep humans accountable for automated decisions
>* Monitor failures, fairness, and changing conditions



In [ ]:
#@title Python Code - Unsafe Automation

# Demonstrate unsafe automation with bridge inspections.
# Compare automated decisions against safer review.
# Keep outputs short for beginners.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a tiny inspection dataset.
bridges = pd.DataFrame({
    "bridge": ["A", "B", "C", "D", "E", "F"],
    "crack_mm": [2, 8, 14, 5, 20, 11],
    "traffic_k": [4, 18, 35, 9, 42, 12],
    "sensor_coverage": [0.95, 0.90, 0.35, 0.80, 0.25, 0.45]
})

# Validate the small table before scoring.
required = {"crack_mm", "traffic_k", "sensor_coverage"}
assert required.issubset(bridges.columns), "Missing required inspection columns"
assert len(bridges) == 6, "Unexpected bridge count"

# Build a simple automated risk score.
bridges["model_score"] = (
    0.06 * bridges["crack_mm"]
    + 0.01 * bridges["traffic_k"]
    + 0.20 * bridges["sensor_coverage"]
)

# Unsafe automation trusts the score alone.
bridges["auto_decision"] = np.where(
    bridges["model_score"] >= 1.10,
    "urgent repair",
    "routine monitoring"
)

# Safer workflow escalates uncertain cases.
bridges["safe_decision"] = np.where(
    bridges["sensor_coverage"] < 0.50,
    "human review",
    bridges["auto_decision"]
)

# Count decisions for a compact summary.
auto_counts = bridges["auto_decision"].value_counts().to_dict()
safe_counts = bridges["safe_decision"].value_counts().to_dict()
review_count = int((bridges["safe_decision"] == "human review").sum())

# Print only key teaching messages.
print("Unsafe automation example: bridge inspection triage")
print(f"Automated decisions: {auto_counts}")
print(f"Safer workflow decisions: {safe_counts}")
print(f"Cases escalated because sensor coverage was low: {review_count}")
print("Lesson: high-stakes predictions need oversight and escalation.")

# Plot score and uncertainty together.
colors = np.where(
    bridges["safe_decision"] == "human review",
    "tomato",
    "steelblue"
)

# Create one compact teaching plot.
plt.figure(figsize=(7, 4))
plt.bar(bridges["bridge"], bridges["model_score"], color=colors)
plt.axhline(1.10, color="black", linestyle="--", label="automation threshold")
plt.xlabel("Bridge identifier")
plt.ylabel("Automated risk score")
plt.title("Unsafe automation hides low data coverage")
plt.legend()
plt.tight_layout()
plt.show()



## **3. Responsible Final Workflow**

### **3.1. Plan Responsible Workflow**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_08/Lecture_B/image_03_01.jpg?v=1783626962" width="250">



>* Connect models to real engineering decisions
>* Define data, uncertainty, limits, and communication

>* Plan data quality across the lifecycle
>* Test models under real field conditions

>* Define review roles, thresholds, and escalation steps
>* Monitor performance and update deployment safeguards



In [ ]:
#@title Python Code - Plan Responsible Workflow

# Plan responsible workflows before deployment.
# Civil models need accountable decisions.
# This checklist scores deployment readiness.

import pandas as pd
import matplotlib.pyplot as plt

# Define workflow checks for a bridge model.
checks = [
    "intended use defined",
    "data quality reviewed",
    "bias and coverage checked",
    "uncertainty communicated",
    "human review assigned",
    "monitoring plan created",
]

# Record simple readiness scores from zero to one.
scores = [1.0, 0.8, 0.5, 0.7, 1.0, 0.4]
workflow = pd.DataFrame({"check": checks, "score": scores})

# Validate the small teaching table.
assert len(workflow) == len(checks)
assert workflow["score"].between(0, 1).all()

# Compute an overall readiness score.
readiness = workflow["score"].mean()
weakest = workflow.loc[workflow["score"].idxmin(), "check"]

# Print concise engineering interpretation.
print("Responsible workflow readiness example")
print(f"Overall readiness score: {readiness:.2f} of 1.00")
print(f"Weakest area needing action: {weakest}")
print("Rule: deploy only with review, limits, and monitoring.")

# Plot readiness across workflow responsibilities.
plt.figure(figsize=(8, 4))
plt.barh(workflow["check"], workflow["score"], color="steelblue")
plt.axvline(0.75, color="red", linestyle="--", label="minimum target")
plt.xlim(0, 1)
plt.xlabel("Readiness score")
plt.title("Responsible Final Workflow Checklist")
plt.legend()
plt.tight_layout()
plt.show()



### **3.2. Ethical Safeguards**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_08/Lecture_B/image_03_02.jpg?v=1783626964" width="250">



>* Build fairness and safety in before deployment
>* Check data, model, decisions, and impacts

>* Check fairness across places, assets, and conditions
>* Protect data privacy and document model limits

>* Use predictions as reviewable decision support
>* Monitor, audit, retrain, or pause models



In [ ]:
#@title Python Code - Ethical Safeguards

# Ethical safeguards support responsible engineering deployment.
# Small audits reveal unequal model performance.
# Human review protects safety and trust.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a tiny bridge inspection example.
np.random.seed(7)
areas = ["urban", "rural", "underserved"]
records = []

# Simulate different error patterns by area.
for area in areas:
    for bridge_id in range(12):
        true_risk = np.random.binomial(1, 0.35)
        miss_rate = {"urban": 0.10, "rural": 0.25, "underserved": 0.40}[area]

        false_alarm = {"urban": 0.12, "rural": 0.18, "underserved": 0.22}[area]
        if true_risk == 1:
            predicted = np.random.binomial(1, 1 - miss_rate)
        else:
            predicted = np.random.binomial(1, false_alarm)

        records.append([area, bridge_id, true_risk, predicted])

# Store results in a small table.
columns = ["area", "bridge_id", "true_risk", "predicted_risk"]
df = pd.DataFrame(records, columns=columns)
assert len(df) == 36 and df.shape[1] == 4

# Compute safety focused audit metrics.
df["missed_high_risk"] = (df.true_risk == 1) & (df.predicted_risk == 0)
df["false_alarm"] = (df.true_risk == 0) & (df.predicted_risk == 1)
summary = df.groupby("area").agg(total=("bridge_id", "count"), high_risk=("true_risk", "sum"), missed=("missed_high_risk", "sum"), false_alarms=("false_alarm", "sum"))

# Convert counts into interpretable rates.
summary["miss_rate"] = summary["missed"] / summary["high_risk"].replace(0, np.nan)
summary["review_required"] = summary["miss_rate"].fillna(0) > 0.25
summary = summary.fillna(0)

# Print a compact ethical safeguard checklist.
print("Ethical safeguard audit for bridge inspection model")
print("Groups audited: urban, rural, underserved")
print("Metric: missed high-risk bridges by area")
for area, row in summary.iterrows():
    print(f"{area}: miss rate {row['miss_rate']:.2f}, review {bool(row['review_required'])}")

# Plot unequal safety performance across areas.
plt.figure(figsize=(7, 4))
plt.bar(summary.index, summary["miss_rate"], color=["steelblue", "orange", "crimson"])
plt.axhline(0.25, color="black", linestyle="--", label="review threshold")
plt.ylabel("Missed high-risk bridge rate")
plt.title("Ethical safeguard: audit errors before deployment")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()



### **3.3. Human Oversight**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_08/Lecture_B/image_03_03.jpg?v=1783626960" width="250">



>* Use models as decision-support tools
>* Keep accountability with qualified human reviewers

>* Escalate uncertain or conflicting model results
>* Question outputs and document human overrides

>* Build oversight throughout deployment
>* Use local insight to guide automation



In [ ]:
#@title Python Code - Human Oversight

# Human oversight keeps automation accountable.
# Engineers review uncertain model recommendations.
# This example builds an oversight checklist.

import pandas as pd
import matplotlib.pyplot as plt

# Create small inspection recommendation records.
records = [
    {"site": "Bridge A", "risk": 0.91, "confidence": 0.88, "equity_flag": False},
    {"site": "Road B", "risk": 0.42, "confidence": 0.51, "equity_flag": True},

    {"site": "Slope C", "risk": 0.76, "confidence": 0.46, "equity_flag": False},
    {"site": "Culvert D", "risk": 0.33, "confidence": 0.82, "equity_flag": False},
]

# Convert records into a compact table.
df = pd.DataFrame(records)
assert len(df) == 4 and "risk" in df.columns

# Define simple oversight rules.
def oversight_action(row):
    if row["confidence"] < 0.60:
        return "Engineer review"

    if row["equity_flag"] and row["risk"] < 0.60:
        return "Bias check"

    if row["risk"] >= 0.80:
        return "Urgent inspection"

    return "Routine monitoring"

# Apply rules before any field action.
df["oversight_action"] = df.apply(oversight_action, axis=1)
summary = df["oversight_action"].value_counts().sort_index()

# Print only concise teaching results.
print("Human oversight decisions for model outputs:")
for site, action in zip(df["site"], df["oversight_action"]):
    print(f"{site}: {action}")

# Plot oversight workload by action type.
plt.figure(figsize=(6, 3))
summary.plot(kind="bar", color="steelblue")
plt.title("Human Oversight Actions")
plt.ylabel("Number of assets")
plt.xlabel("Required human step")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Responsible Deployment**</font>


In this lecture, you learned to:
- Evaluate competing models using metrics, plots, interpretability, and engineering consequences. 
- Identify risks involving bias, imbalance, explainability, data drift, and unsafe automation. 
- Design a responsible final machine learning workflow for a civil engineering use case. 

<font color='yellow'>Congratulations on completing this course!</font>